# 06 — Gate AB-1: S0 anchors × budget levels × climate levels, LP twins, S4 pilot (R)

Kernel `R (y2y)`; live internet (Gurobi WLS). Mirrors the parent's Gate-4 engine artifacts
(`12_gate4_ensemble`: certified binary anchor + Gurobi-proportion twin per formulation), on the AB
stack, for the AB-1 set:

| formulation | level | climate | why |
|---|---|---|---|
| `s0_ssp585_theta5` | A and B | 585 | the reference formulation at both budget levels (D-AB5 v2) |
| `s0_ssp245_theta5` | A and B | 245 | climate axis (weights re-derived at 245 by 05) |
| `s4_ssp585_theta<θ>` | A | 585 | the S4 pilot: does the ladder target bind at the kink? (M4.3) |

Each → `runs/ab_l/<level>/<formulation_id>/anchor/` (binary, opt_gap 1e-4, NumericFocus) and
`.../twin/` (proportion). Resumable per artifact. 07 re-solves the anchor inside the MGA machinery
and checks it against these certificates (drift ≤ 1e-3, parent convention).

In [1]:
# ---- setup: engine, AB manifest, frozen inputs -------------------------------------------------------
ANALYSIS <- "ab_y2y"
PROJ <- normalizePath(getwd())
while (!file.exists(file.path(PROJ, "config.py"))) {
  parent <- dirname(PROJ)
  if (identical(parent, PROJ)) stop("config.py not found above getwd() -- open from inside the repo")
  PROJ <- parent
}
setwd(PROJ)
source(file.path(PROJ, "prioritizr_core.R"))
HERE <- file.path(PROJ, "analyses", "alberta_prioritization")
AB_DIR <- file.path(PROJ, "input_data", "aligned_stack_ab")
py <- file.path(PROJ, ".venv", "bin", "python")
code <- paste0("import config; print(config.write_manifest(analysis='", ANALYSIS, "', ",
               "handoff_dir=config.AB_HANDOFF_DIR, manifest_path=config.AB_HANDOFF_DIR/'manifest.json'))")
out <- suppressWarnings(system2(py, c("-c", shQuote(code)), stdout = TRUE, stderr = TRUE))
if (!is.null(attr(out, "status")) && attr(out, "status") != 0) stop("AB manifest refresh FAILED:\n  ", paste(out, collapse = "\n  "))
mpath <- file.path(AB_DIR, "manifest.json")

SC  <- jsonlite::read_json(file.path(HERE, "spec", "scenarios_ab_v1.json"))
LV  <- jsonlite::read_json(file.path(HERE, "spec", "ab_budget_levels_v1.json"))$levels
EXT <- jsonlite::read_json(file.path(HERE, "spec", "ab_extent_v1.json"))
TH_S4 <- SC$`_meta`$s4_ladder$chosen_theta
T_S4  <- SC$`_meta`$s4_ladder$chosen_target
REAL245 <- "input_data/aligned_stack_ab/climate_realizations/macrorefugia_245_2071_2100.tif"
RUNS_REL <- "analyses/alberta_prioritization/runs/ab_l"
cat(sprintf("scenarios_ab_v1 derived %s | S4 theta %sx -> t = %.3f | levels: A %d cells (%.1f%%), B %d cells (%.1f%%)\n",
            substr(SC$`_meta`$derived_utc, 1, 19), TH_S4, T_S4,
            LV$A$budget_cells, 100 * LV$A$budget_pct, LV$B$budget_cells, 100 * LV$B$budget_pct))

# the AB-1 formulation set
FORMS <- list(
  list(id = "s0_ssp585_theta5", scen = "S0_balanced", climate = "ssp585", levels = c("A", "B")),
  list(id = "s0_ssp245_theta5", scen = "S0_balanced", climate = "ssp245", levels = c("A", "B")),
  list(id = sprintf("s4_ssp585_theta%s", sub("\\.0$", "", as.character(TH_S4))), scen = "S4_carbon", climate = "ssp585", levels = c("A"))
)
wt_for <- function(f) list(
  w = if (f$climate == "ssp245") SC[[f$scen]]$weights_ssp245 else SC[[f$scen]]$weights,
  t = SC[[f$scen]]$targets)

scenarios_ab_v1 derived 2026-09-03T20:08:50 | S4 theta 2x -> t = 0.772 | levels: A 38055 cells (44.7%), B 33014 cells (38.8%)


In [2]:
# ---- base contexts: ingested once per climate level; budget applied per level below ------------------
ctx585 <- pr_setup(mpath, PROJ)
ctx585 <- modifyList(ctx585, pr_ingest(ctx585))
ctx245 <- pr_setup(mpath, PROJ)
ctx245$layers$path[ctx245$layers$name == "climate_type_macrorefugia"] <- REAL245
ctx245 <- modifyList(ctx245, pr_ingest(ctx245))
base_ctx <- function(f, level) {
  b <- if (f$climate == "ssp245") ctx245 else ctx585
  b <- pr_override(b, budget_pct = LV[[level]]$budget_pct,
                   results_dir = file.path(RUNS_REL, level, f$id), results_subdir = "_base")
  b <- modifyList(b, pr_planning_units(b))
  stopifnot(b$n_pu == EXT$n_pu, b$n_locked == EXT$n_locked,
            abs(round(b$budget) - LV[[level]]$budget_cells) <= 1)
  b
}
cat("base contexts ingested (585 canonical; 245 realization patched)\n")

prioritizr 8.1.0 | terra 1.9.34 | analysis=ab_y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/ab_y2y
ingested 35 features (8 continuous + 27 EFG) + cost + PA mask | grid 1286 x 3312 @ 1000 m
normalized 35 features to total=100000 each (scale-invariant conditioning)
prioritizr 8.1.0 | terra 1.9.34 | analysis=ab_y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/ab_y2y
ingested 35 features (8 continuous + 27 EFG) + cost + PA mask | grid 1286 x 3312 @ 1000 m
normalized 35 features to total=100000 each (scale-invariant condit

In [3]:
# ---- runner: one engine artifact (anchor = binary certified; twin = proportion) --------------------------
run_artifact <- function(f, level, artifact) {
  done <- file.path(PROJ, RUNS_REL, level, f$id, artifact, "run_summary.json")
  if (file.exists(done)) { cat(sprintf("   %s/%s/%s exists -- skipped\n", level, f$id, artifact)); return(invisible(NULL)) }
  ov <- if (artifact == "anchor") list(solver = "gurobi", decision_type = "binary", opt_gap = 1e-4, portfolio_n = 1)
        else                      list(solver = "gurobi", decision_type = "proportion", opt_gap = 1e-4, portfolio_n = 1)
  wt <- wt_for(f)
  actx <- do.call(pr_override, c(list(base_ctx(f, level), targets = wt$t, feature_weight_multipliers = wt$w,
                                      results_subdir = artifact), ov))
  actx <- modifyList(actx, pr_weights(actx))
  actx <- modifyList(actx, pr_targets(actx))
  actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  sv <- pr_solve(actx); actx$s <- sv$s; actx$timing <- sv$timing
  actx$n_sol <- sv$n_sol; actx$sol_attrs <- sv$sol_attrs
  actx <- modifyList(actx, pr_summaries(actx))
  pr_write_outputs(actx)
  jsonlite::write_json(list(formulation_id = f$id, scenario = f$scen, climate_level = f$climate, level = level,
                            budget_cells = LV[[level]]$budget_cells, budget_pct = LV[[level]]$budget_pct,
                            weights = wt$w, targets = wt$t, artifact = artifact,
                            created_utc = format(Sys.time(), tz = "UTC")),
                       file.path(PROJ, RUNS_REL, level, f$id, artifact, "artifact_meta.json"),
                       auto_unbox = TRUE, pretty = TRUE, digits = 10)
  invisible(NULL)
}

t_batch <- proc.time()[["elapsed"]]
for (f in FORMS) for (level in f$levels) {
  cat(sprintf("\n===================== %s @ level %s =====================\n", f$id, level))
  run_artifact(f, level, "anchor")
  run_artifact(f, level, "twin")
  cat(sprintf("== elapsed %.1f min\n", (proc.time()[["elapsed"]] - t_batch) / 60))
}
cat("\nAB-1 ARTIFACTS COMPLETE -- next: 07_ab2_mga.ipynb (R), then 08_ab2_analysis.ipynb (py)\n")


===================== s0_ssp585_theta5 @ level A =====================
  override budget_pct       -> 0.4470064
  override results_dir      -> analyses/alberta_prioritization/runs/ab_l/A/s0_ssp585_theta5
  override results_subdir   -> _base
  EFFECTIVE targets: <none> | weight multipliers: <none>
  outputs  -> analyses/alberta_prioritization/runs/ab_l/A/s0_ssp585_theta5/_base
planning units: 85,133 cells | budget = 45% = 38,055 cells
locked-in [pa_mask]: 27,972 cells (32.9% of window) -- fits within budget
  override targets          -> irrecoverable_carbon_m_soc=0.322
  override feature_weight_multipliers -> climate_type_macrorefugia=1.038, transboundary_connectivity=0.30626, climate_corridors=1.5978, irrecoverable_carbon_m_soc=0.219191, irrecoverable_carbon_biomass=0.171899, aoh_richness_birds=1.38582, aoh_richness_mammals=2.28103
  override results_subdir   -> anchor
  override solver           -> gurobi
  override decision_type    -> binary
  override opt_gap          -> 1e-04
  o

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (35 total)
│└•planning units:
│ ├•data:       <SpatRaster> (85133 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 38055)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.322 and 1)
││└•weights:    continuous values (between 0.03703704 and 2.281027)
│├•constraints: 
││└•1:          locked in constraints (27972 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 36 rows, 85168 columns and 1194296 nonzeros (Min)
Model fingerprint: 0x63e13a9e
Model has 35 linear objective coefficients
Variable types: 35 continuous, 85133 integer (85133 binary)
Coefficient statistics:
  Matrix range     [1e-03, 1e+05]
  Objective range  [4e-02, 2e+00]
  Bounds range     [1e+00, 1e+00]
  RH

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (35 total)
│└•planning units:
│ ├•data:       <SpatRaster> (85133 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 38055)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.322 and 1)
││└•weights:    continuous values (between 0.03703704 and 2.281027)
│├•constraints: 
││└•1:          locked in constraints (27972 planning units)
│└•decisions:   proportion decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 36 rows, 85168 columns and 1194296 nonzeros (Min)
Model fingerprint: 0x1b4e2de8
Model has 35 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e-03, 1e+05]
  Objective range  [4e-02, 2e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e+04, 1e+05]

Presolve removed 32 rows and 

...

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 36 rows, 85168 columns and 1194296 nonzeros (Min)
Model fingerprint: 0x73ff9678
Model has 35 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e-03, 1e+05]
  Objective range  [4e-02, 2e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e+04, 1e+05]

Presolve removed 32 rows and 

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (35 total)
│└•planning units:
│ ├•data:       <SpatRaster> (85133 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 38055)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.772 and 1)
││└•weights:    continuous values (between 0.03703704 and 2.053178)
│├•constraints: 
││└•1:          locked in constraints (27972 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 36 rows, 85168 columns and 1194296 nonzeros (Min)
Model fingerprint: 0x344b96a0
Model has 35 linear objective coefficients
Variable types: 35 continuous, 85133 integer (85133 binary)
Coefficient statistics:
  Matrix range     [1e-03, 1e+05]
  Objective range  [4e-02, 2e+00]
  Bounds range     [1e+00, 1e+00]
  RH

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (35 total)
│└•planning units:
│ ├•data:       <SpatRaster> (85133 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 38055)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.772 and 1)
││└•weights:    continuous values (between 0.03703704 and 2.053178)
│├•constraints: 
││└•1:          locked in constraints (27972 planning units)
│└•decisions:   proportion decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 36 rows, 85168 columns and 1194296 nonzeros (Min)
Model fingerprint: 0x741da0c2
Model has 35 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e-03, 1e+05]
  Objective range  [4e-02, 2e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [4e+04, 1e+05]

Presolve removed 31 rows and 

In [4]:
# ---- read-back: anchor objective / twin objective (LP <= MILP) / m_soc capture / time -----------------------
obj_of <- function(rs) tryCatch(as.numeric(unlist(rs$solver_provenance$objective))[1], error = function(e) NA)
for (f in FORMS) for (level in f$levels) {
  d <- file.path(PROJ, RUNS_REL, level, f$id)
  a <- file.path(d, "anchor", "run_summary.json"); tw <- file.path(d, "twin", "run_summary.json")
  if (!file.exists(a)) next
  ra <- jsonlite::read_json(a); rep <- read.csv(file.path(d, "anchor", "portfolio_representation.csv"))
  ms <- rep$relative_held[rep$feature == "irrecoverable_carbon_m_soc"]
  ot <- if (file.exists(tw)) obj_of(jsonlite::read_json(tw)) else NA
  oa <- obj_of(ra)
  cat(sprintf("%-20s %s  anchor %.6f (%4.0f s) | twin %s [LP<=MILP %s] | m_soc capture %.4f | budget %s\n",
              f$id, level, oa, ra$solve_seconds, ifelse(is.na(ot), "--", sprintf("%.6f", ot)),
              ifelse(is.na(ot), "?", ifelse(ot <= oa + 1e-6, "OK", "VIOLATED")), ms,
              format(ra$budget_cells, big.mark = ",")))
}

s0_ssp585_theta5     A  anchor 4.460900 (   1 s) | twin 4.460900 [LP<=MILP OK] | m_soc capture 0.7552 | budget 38,055
s0_ssp585_theta5     B  anchor 5.022000 (   1 s) | twin 5.022000 [LP<=MILP OK] | m_soc capture 0.7324 | budget 33,014
s0_ssp245_theta5     A  anchor 4.452200 (   1 s) | twin 4.452200 [LP<=MILP OK] | m_soc capture 0.7533 | budget 38,055
s0_ssp245_theta5     B  anchor 5.013000 (   1 s) | twin 5.013000 [LP<=MILP OK] | m_soc capture 0.7322 | budget 33,014
s4_ssp585_theta2     A  anchor 4.288300 (   2 s) | twin 4.288300 [LP<=MILP OK] | m_soc capture 0.7720 | budget 38,055
